In [ ]:
from datasets import load_dataset
import pandas as pd
from collections import defaultdict, Counter
import math
import numpy as np
import torch
import torch.nn as nn
import json
import re
import kenlm
from tqdm import tqdm
import math
from itertools import chain

C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load dataset
dataset = load_dataset("yammdd/vietnamese-error-correction-corpus")
df_train = pd.DataFrame(dataset['train'])

In [4]:
vocab = []
with open(r"D:\NLP\project\vocabulary.txt", 'r', encoding='utf-8') as f:
    vocab = f.read().splitlines()

# Lọc các từ giống nhau
vocab = list(dict.fromkeys(vocab))

# Ánh xạ word và idx
word_to_idx = {word: i for i, word in enumerate(vocab)}

In [5]:
counts_1 = Counter() 
counts_2 = Counter() 
counts_3 = Counter() 

for sentence in df_train['target']:
    sentence = sentence.lower()
    sentence = re.sub(r'\s+([.,!?;:\)\]\}])', r'\1', sentence)

    # Làm sạch cơ bản và tách từ theo khoảng trắng
    tokens = str(sentence).lower().split()
    if not tokens:
        continue
        
    # Đếm từ đơn 
    counts_1.update(tokens)
    
    # Đếm từ ghép 2 bằng cách trượt cửa sổ cặp đôi
    # zip(tokens, tokens[1:]) tạo ra các cặp liên tiếp
    bigrams = [" ".join(p) for p in zip(tokens, tokens[1:])]
    counts_2.update(bigrams)
    
    # Đếm từ ghép 3 bằng cách trượt cửa sổ bộ ba
    trigrams = [" ".join(t) for t in zip(tokens, tokens[1:], tokens[2:])]
    counts_3.update(trigrams)

In [6]:
with open(r"D:\NLP\project\vietnamese-stopwords.txt", 'r', encoding='utf-8') as f:
    stopwords = f.read().splitlines()
stopword = set(stopwords)

In [ ]:
with open(r"D:\NLP\project\telex.txt", "r", encoding="utf-8") as f:
    telex = f.read()

telex = re.sub(r',\s*}', '\n}', telex)
telex = json.loads(telex)

In [8]:
def create_telex_form(word):
    word = word.lower()
    prefix = ""      # Phụ âm đầu
    vowel_base = ""  # Nguyên âm gốc
    suffix = ""      # Phụ âm cuối
    word_tone = ""   # Dấu thanh
    word_mod = ""    # Ký tự gõ mũ/móc

    VOWELS = "aeiouy" # Các nguyên âm tiếng Việt
    state = 0 # 0: phụ âm đầu, 1: nguyên âm

    i = 0
    while i < len(word):
        step = 1
        # Trường hợp 'ươ'
        if i < len(word) - 1 and word[i:i+2] in telex:
            char = word[i:i+2]
            step = 2
        else:
            char = word[i]

        # Nếu ký tự nằm trong từ điển
        if char in telex:
            if char == 'đ':
                if state == 0: prefix += 'dd'
                else: suffix += 'dd'
            else:
                vowel_base += telex[char][0]
                if telex[char][1]: word_mod = telex[char][1]
                if telex[char][2]: word_tone = telex[char][2]
                state = 1

        # Nếu ký tự là chữ bình thường
        else:
            if char in VOWELS:
                vowel_base += char
                state = 1
            else:
                if state == 0: prefix += char
                else: suffix += char

        i += step

    # Tạo biến thể
    variants = set()
    inline_vowel = vowel_base + word_mod

    # Kiểu 1 & 2: Gõ chuẩn ngay sau nguyên âm hoặc ném dấu thanh ra cuối
    variants.add(prefix + inline_vowel + word_tone + suffix)
    variants.add(prefix + inline_vowel + suffix + word_tone)

    # Kiểu 3: Phím bổ nghĩa (w, a, e, o) ném ra cuối từ
    if word_mod:
        variants.add(prefix + vowel_base + word_tone + suffix + word_mod)
        variants.add(prefix + vowel_base + suffix + word_mod + word_tone)
        variants.add(prefix + vowel_base + suffix + word_tone + word_mod)

    # Trường hợp đặc biệt của "ươ"
    if vowel_base == 'uo' and word_mod == 'w':
        # Tách w hai lần ngay sau nguyên âm
        variants.add(prefix + 'uwow' + word_tone + suffix)
        variants.add(prefix + 'uwow' + suffix + word_tone)

        # Tách w đầu, w cuối
        variants.add(prefix + 'uwo' + word_tone + suffix + 'w')
        variants.add(prefix + 'uwo' + suffix + 'w' + word_tone)
        variants.add(prefix + 'uwo' + suffix + word_tone + 'w')

    return list(v for v in variants if v)

In [9]:
def get_deletes(word, k = 2):
    queue = {word}
    variant_list = set()
    
    for _ in range(k):
        temp_queue = set()
        for w in queue:
            if len(w) > 1:
                # Tạo deletes cho vòng hiện tại
                deletes = {w[:i] + w[i+1:] for i in range(len(w))}
                variant_list.update(deletes)
                temp_queue.update(deletes)
        queue = temp_queue
    return variant_list

In [10]:
# Tạo danh sách các từ thiếu từ 0 đến k kí tự trong vocab
# Tạo 1 Dictionary (Hash Map) để lưu toàn bộ biến thể xóa
sym_dict = defaultdict(list)

for word in vocab:
    length = word.split(' ')
    if len(length) > 1:
        continue
    # Lấy các biến thể và từ gốc của 1 từ
    base_forms = [word] + create_telex_form(word)

    for form in base_forms:
        # Lưu form này (distance 0)
        if word not in sym_dict[form]:
            sym_dict[form].append(word)

        # Tạo deletes cho từng form và lưu vào từ điển
        variant_list = get_deletes(form)
        for variant in variant_list:
            # Map biến thể tới từ gốc 'word' hiện tại nếu chưa map
            if word not in sym_dict[variant]:
                sym_dict[variant].append(word)

In [ ]:
# Hàm tính khoảng cách của xâu 1 và xâu 2 bằng thuật toán Damerau-Levenshtein
# Là hàm edit_distance nhưng có thêm phép đổi chỗ các kí tự (gõ lộn thứ tự)
# Cải thiện thêm bằng khoảng cách bàn phím cho trường hợp gõ nhầm

# Các phím liền kề trên bàn phím để tính trọng số
ADJACENT_KEYS = {
    'q': 'wea', 'w': 'qeasd', 'e': 'wrsdf', 'r': 'etdfg', 't': 'ryfgh', 'y': 'tughj', 'u': 'yihjk', 'i': 'uojkl', 'o': 'ipkl', 'p': 'ol',
    'a': 'qwsz', 's': 'weadzx', 'd': 'ersfxc', 'f': 'rtdgcv', 'g': 'tyfhvb', 'h': 'yugjbn', 'j': 'uihknm', 'k': 'iojlm', 'l': 'opk',
    'z': 'asx', 'x': 'sdzc', 'c': 'dfxv', 'v': 'fgcb', 'b': 'ghvn', 'n': 'hjbm', 'm': 'jkn'
}

# Các cặp âm dễ nhầm lẫn trong phát âm tiếng Việt (Lỗi ngữ âm)
CONFUSION_PAIRS = {
    ('s', 'x'), ('x', 's'),
    ('l', 'n'), ('n', 'l'),
    ('d', 'r'), ('r', 'd'),
    ('d', 'gi'), ('gi', 'd'),
    ('i', 'y'), ('y', 'i'),
    ('c', 'k'), ('c', 'k'),
    ('ch', 'tr'), ('tr', 'ch')
}

def edit_distance(s1, s2):
    n, m = len(s1), len(s2)
    dp = [[0.0] * (m + 1) for _ in range(n + 1)]

    # Khởi tạo giá trị hàng và cột đầu tiên
    for i in range(n + 1):
        dp[i][0] = i

    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            char1 = s1[i - 1]
            char2 = s2[j - 1]
            # Tính chi phí thay thế (0 nếu giống nhau, 0.5 nếu khác mà gần nhau trên bàn phím, 1 nếu khác và ở xa trên bàn phím)
            if char1 == char2:
                sub_cost = 0.0
            elif (char1, char2) in CONFUSION_PAIRS:
            # Lỗi phát âm vùng miền (VD: s vs x). Phạt rất nhẹ vì đây là lỗi cực kỳ phổ biến.
                sub_cost = 0.4
            elif char1 in ADJACENT_KEYS.get(char2, "") or char2 in ADJACENT_KEYS.get(char1, ""):
                # Nếu gõ nhầm 2 phím cạnh nhau (VD: a và s), chi phí chỉ là 0.6
                sub_cost = 0.5
            else:
                # Lỗi gõ nhầm phím xa nhau, chi phí 1.0
                sub_cost = 1

            # Tính chi phí thêm / xóa
            del_cost = 1
            ins_cost = 1

            dp[i][j] = min(
                dp[i - 1][j] + del_cost,       # Xóa
                dp[i][j - 1] + ins_cost,       # Thêm
                dp[i - 1][j - 1] + sub_cost    # Thay thế
            )

            # Phép Đổi chỗ (Transposition)
            if i > 1 and j > 1 and s1[i - 1] == s2[j - 2] and s1[i - 2] == s2[j - 1]:
                dp[i][j] = min(dp[i][j], dp[i - 2][j - 2] + 0.5)

            if i >= 2 and j >= 2:
                sub1 = s1[i-2:i]
                sub2 = s2[j-2:j]
                if (sub1, sub2) in CONFUSION_PAIRS:
                    dp[i][j] = min(dp[i][j], dp[i-2][j-2] + 0.4)
            
            if i >= 2 and j >= 1:
                sub1 = s1[i-2:i]
                sub2 = s2[j-1:j]
                if (sub1, sub2) in CONFUSION_PAIRS:
                    dp[i][j] = min(dp[i][j], dp[i-2][j-1] + 0.4)

            if i >= 1 and j >= 2:
                sub1 = s1[i-1:i]
                sub2 = s2[j-2:j]
                if (sub1, sub2) in CONFUSION_PAIRS:
                    dp[i][j] = min(dp[i][j], dp[i-1][j-2] + 0.4)

    return dp[n][m]

In [12]:
def edit_distance_telex(s1, s2):
    min_dist = float('inf')

    string1 = create_telex_form(s1)
    string2 = create_telex_form(s2)

    for str1 in string1:
        for str2 in string2:
            dist = edit_distance(str1, str2)
            if dist < min_dist:
                min_dist = dist
    return min_dist

In [13]:
# Hàm tìm từ gần nhất với từ nhập vào, hàm trả về các từ gần nhất và khoảng cách của nó
def lookup(word, k=2):
    variant_list = [word] + list(get_deletes(word))
    for telex in create_telex_form(word):
        variant_list += list(get_deletes(telex))

    # Lưu đáp án là các từ gần nhất và khoảng cách của nó
    candidates = {}

    # Tra cứu các biến thể
    for variant in variant_list:
        if variant in sym_dict:
            for suggestion in sym_dict[variant]:
                if suggestion in candidates:
                    continue

                dist = edit_distance_telex(word, suggestion)

                # Nhận khi khoảng cách <= k, tồn tại trong từ điển và có tần suất > 0
                if dist <= k and suggestion in vocab and counts_1.get(suggestion, 0) > 0:
                    candidates[suggestion] = (dist, counts_1.get(suggestion, 0))

    # Xếp hạng ưu tiên: Distance nhỏ trước -> Tần suất cao trước
    result = sorted(candidates.items(),
                    key=lambda x: (x[1][0], -x[1][1]))

    ans = []
    # Giải nén thẳng tuple cho dễ đọc, đổi tên biến tránh ghi đè tham số 'word'
    for cand_word, (dist, count) in result:
        ans.append(cand_word)

    return ans

In [14]:
model_lm = kenlm.Model(r"D:/NLP/project/trigram.bin")

def detect_error_word(sentence):
    scores = list(model_lm.full_scores(sentence))[:-1]
    words = sentence.split()

    error_indices = set()
    valid_probs = []
    valid_indices = []

    for i, (prob, length, is_oov) in enumerate(scores):
        if i >= len(words):
            continue
            
        # Lọc số
        if re.search(r'[0-9]', words[i]):
            continue

        # Lọc các từ out of vocabulary
        if is_oov:
            error_indices.add(i)
            
        valid_probs.append(prob)
        valid_indices.append(i)

    # Ngưỡng ken_lm
    if valid_probs:
        mean_prob = np.mean(valid_probs)
        std_prob = np.std(valid_probs) 
        
        alpha = 1.4
        dynamic_threshold = mean_prob - (alpha * std_prob)
        
        hard_ceiling = -5.52
        
        hard_floor = -5.93

        for idx, prob in zip(valid_indices, valid_probs):
            is_anomaly = (prob < dynamic_threshold) and (prob < hard_ceiling)
            is_absolute_error = (prob < hard_floor)
            
            if is_anomaly or is_absolute_error:
                error_indices.add(idx)

    return sorted(list(error_indices))

model Skipgram

In [15]:
class SkipGram(nn.Module):

    def __init__(self, vocab_size, embed_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):

        embed = self.embedding(x)
        out = self.linear(embed)

        return out

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBED_DIM = 300
VOCAB_SIZE = len(vocab)

model_skipgram = SkipGram(VOCAB_SIZE, EMBED_DIM).to(device)

model_path = r"D:/NLP/project/model_skipgram.pth"
model_skipgram.load_state_dict(torch.load(model_path, weights_only=True))

<All keys matched successfully>

In [18]:
def find_misspelled_words_and_targets(input_sentence, target_sentence):
    # GỢI Ý: Nếu thuật toán của bạn dựa trên việc tách từ bằng khoảng trắng:
    input_tokens = input_sentence.split()
    target_tokens = target_sentence.split()
    
    error_indices = []
    pairs = []
    # Giả định số từ bằng nhau (nếu lệch từ do thừa/thiếu ký tự dấu cách, 
    # bạn cần dùng thuật toán alignment hoặc Diff của python)
    if len(input_tokens) != len(target_tokens):
        return [], []

    for i in range(len(input_tokens)):
        if re.search(r'[0-9]', input_tokens[i]) or re.search(r'[0-9]', target_tokens[i]):
            continue

        if input_tokens[i] != target_tokens[i] and target_tokens[i] in word_to_idx:
            pairs.append((input_tokens[i], target_tokens[i]))
            error_indices.append(i)
            
    return pairs, error_indices

In [ ]:
embeddings = model_skipgram.embedding.weight.data
embedding_matrix = embeddings.cpu().numpy()

norms = np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1 
norm_embedding_matrix = embedding_matrix / norms

def extract_candidates_and_features(error_word, sentence_words, error_idx, error_indices, window_size=3):
    n_words = len(sentence_words)

    if error_idx >= n_words or error_idx < 0:
        return []
    
    local_start = max(0, error_idx - window_size)
    local_end = min(n_words, error_idx + window_size + 1)
    
    prefix_words = sentence_words[local_start:error_idx]
    suffix_words = sentence_words[error_idx + 1:local_end]
    
    prefix_str = " ".join(prefix_words) + " " if prefix_words else ""
    suffix_str = " " + " ".join(suffix_words) if suffix_words else ""

    start = max(0, error_idx - window_size)
    end = min(n_words, error_idx + window_size + 1)
    
    valid_context_words = []
    for i in range(start, end):
        if i == error_idx:
            continue
        
        if (i < error_idx or i not in error_indices) and sentence_words[i] not in stopwords:
            word = sentence_words[i]
            if word in word_to_idx:
                dist_weight = 1.0 / abs(i - error_idx) 
                valid_context_words.append((word, dist_weight))


    prev_word = sentence_words[error_idx - 1].lower() if error_idx > 0 else "<s>"
    prev_2_word = sentence_words[error_idx - 2].lower() if error_idx > 1 else "<s>"
    
    next_word = sentence_words[error_idx + 1].lower() if error_idx < n_words - 1 else "</s>"
    next_2_word = sentence_words[error_idx + 2].lower() if error_idx < n_words - 2 else "</s>"

    ctx_indices = []
    ctx_weights = []
    for ctx_word, weight in valid_context_words:
        ctx_indices.append(word_to_idx[ctx_word])
        ctx_weights.append(weight)

    mock_candidates = []
    top = []
    
    candidates = lookup(error_word)
    valid_candidates = [c for c in candidates if c in word_to_idx]
    

    cand_to_sim = {}
    if valid_candidates and ctx_indices:
        cand_indices = [word_to_idx[c] for c in valid_candidates]
        
        C = norm_embedding_matrix[cand_indices]
        W = norm_embedding_matrix[ctx_indices]
        
        S = np.dot(C, W.T)
        weights_array = np.array(ctx_weights)
        S_weighted = S * weights_array
        
        max_sims = np.max(S_weighted, axis=1)
        cand_to_sim = {cand: max_sims[i] for i, cand in enumerate(valid_candidates)}


    for rank_idx, candidate in enumerate(candidates):
        candidate_lower = candidate.lower()
        
        # 1. Similarity
        weighted_sim = cand_to_sim.get(candidate, 0.0)
        norm_sim = max(0.0, weighted_sim) 

        # 2. KenLM 
        local_sentence_str = f"{prefix_str}{candidate}{suffix_str}".strip()
        ken_score = model_lm.score(local_sentence_str)
        norm_ken = max(0.0, (ken_score + 15.0) / 15.0) 

        # 3. N-gram : Tổng hợp năng lượng tần suất thực tế
        count_val_1 = counts_1.get(candidate, 0)
        
        # GỘP BIGRAM: Tổng xung lực trái + phải
        c2_left  = counts_2.get(f"{prev_word} {candidate_lower}", 0)
        c2_right = counts_2.get(f"{candidate_lower} {next_word}", 0)
        count_val_2 = c2_left + c2_right
        
        # GỘP TRIGRAM: Tổng xung lực 3 hướng dịch chuyển toàn diện
        c3_center = counts_3.get(f"{prev_word} {candidate_lower} {next_word}", 0)
        c3_left   = counts_3.get(f"{candidate_lower} {next_word} {next_2_word}", 0)
        c3_right  = counts_3.get(f"{prev_2_word} {prev_word} {candidate_lower}", 0)
        count_val_3 = c3_center + c3_left + c3_right

        # Chuẩn hóa log mượt dữ liệu cho màng lọc tuyển chọn Hard Negatives
        norm_count_1 = min(1.0, math.log1p(count_val_1) / 15.0)
        norm_count_2 = min(1.0, math.log1p(count_val_2) / 12.0)
        norm_count_3 = min(1.0, math.log1p(count_val_3) / 12.0) 

        # 4. Edit Distance
        dist = edit_distance_telex(error_word, candidate)
        norm_edit = 1.0 / (dist + 1) 

        # 5. Length ratio
        len_err = len(error_word)
        len_cand = len(candidate)
        length_ratio = min(len_err, len_cand) / max(len_err, len_cand) if max(len_err, len_cand) > 0 else 0

        total_score = (
            (0.30 * norm_ken) +    
            (0.25 * norm_edit) +  
            (0.10 * length_ratio) + 
            (0.20 * norm_count_2) + 
            (0.05 * norm_count_3) +     
            (0.05 * norm_count_1) +
            (0.05 * norm_sim)      
        )
        
        top.append((total_score, candidate, ken_score, weighted_sim, count_val_1, 
                    count_val_2, count_val_3, dist, length_ratio, rank_idx))

    # Thực hiện Hard Negative Mining để lấy ra các ca cạnh tranh cao nhất
    top.sort(key=lambda x: x[0], reverse=True)
    
    for item in top:
        (_, candidate, ken_score, weighted_sim, c1, c2, c3, dist_val, length_ratio) = item
        
        feature_vector = [ken_score, weighted_sim, c1, c2, c3, dist_val, length_ratio,]
        mock_candidates.append((candidate, feature_vector))

    return mock_candidates

## Lấy feature ở tập df_train

In [ ]:
embeddings = model_skipgram.embedding.weight.data
embedding_matrix = embeddings.cpu().numpy()

# Đọc file dữ liệu lỗi chính tả đã được lọc từ bước trước
df = pd.read_csv("loi_chinh_ta.csv")

X_train_list = []
y_train_list = []
group_train_list = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Trích xuất đặc trưng"):
    input_sent = str(row['input']).lower()
    target_sent = str(row['target']).lower()

    input_sent = re.sub(r'[^\w\s_]', '', input_sent)
    target_sent = re.sub(r'[^\w\s_]', '', target_sent)
    
    error_pairs, error_indices = find_misspelled_words_and_targets(input_sent, target_sent)
    
    if not error_pairs:
        continue 
        
    sentence_words = input_sent.split()
        
    for i in range(len(error_indices)):
        error_word, correct_word = error_pairs[i]

        candidates_with_scores = extract_candidates_and_features(
            error_word, 
            sentence_words, 
            error_indices[i], 
            error_indices
        )
        
        if not candidates_with_scores:
            continue 

        positives = []
        negatives = []

        for candidate_word, feature_vector in candidates_with_scores:
            if candidate_word == correct_word:
                positives.append((candidate_word, feature_vector))
            else:
                negatives.append((candidate_word, feature_vector))


        if not positives:
            continue 
        
        max_negatives = 20
        hard_negatives = negatives[:max_negatives]

        final_candidates = positives + hard_negatives

        group_X = []
        group_y = []

        for cand, feat in final_candidates:
            label = 1 if cand == correct_word else 0
            group_X.append(feat)
            group_y.append(label)

        X_train_list.extend(group_X)
        y_train_list.extend(group_y)

        group_train_list.append(len(final_candidates))

        if error_indices[i] < len(sentence_words):
            sentence_words[error_indices[i]] = correct_word

In [ ]:
X_train = np.array(X_train_list)
y_train = np.array(y_train_list)
group_train = np.array(group_train_list)

print(X_train.shape)
print(y_train.shape)
print(len(group_train))

In [ ]:
np.savez_compressed(
    'dataset_spell_correction.npz', 
    X_train=X_train, 
    y_train=y_train, 
    group_train=group_train
)